In [1]:
#### Libraries ###
import os
import glob
import pandas as pd
import numpy as np
import scanpy as sc

import scipy.sparse as sp  
from scipy.sparse import csr_matrix
from scipy.io import mmread


import matplotlib.pyplot as plt

os.chdir("/data/scottaa/cta_onco_fetal")

# Embryos


In [15]:
sample_list = []
for file in os.listdir("datasets/embryos_mixed/original_data"):
    sample_list.append(file.split("_")[0]+"_"+file.split("_")[1])

sample_list

['F_10W',
 'F_11W',
 'F_12W',
 'F_14W',
 'F_18W',
 'F_20W',
 'F_23W',
 'F_24W',
 'F_26W',
 'F_5W',
 'F_7W',
 'F_8W',
 'M_10W',
 'M_12W',
 'M_19W',
 'M_20W',
 'M_21W',
 'M_25W',
 'M_4W',
 'M_9W']

In [42]:

    

project_name = "embryos_mixed"
sample_id = "F_10W"
original_data_dir = "datasets/embryos_mixed/original_data"
adata_init_path =  os.path.join("datasets/embryos_mixed/raw_data", f"{sample_id}_init.h5ad")
if project_name in ["fetal_gonad", "embryos_mixed"]:
    sample_meta_path = "datasets/embryos_mixed/embryos_mixed_concat_sample_meta.csv"
    sample_meta_df = pd.read_csv(sample_meta_path)
def import_raw_data_csv(project_name, sample_id, original_data_dir,adata_init_path, sample_meta_df):
    """
    ["embryos_mixed"]
    """
    if project_name in ["embryos_mixed"]:
        raw_counts_path = os.path.join(original_data_dir, f"{sample_id}_concat_gene_expression.txt")
    df = pd.read_csv(raw_counts_path, index_col = 0).T
    
    adata = sc.AnnData(df)
    adata.var_names_make_unique()

   
        
    adata.obs["sample_id"] = sample_id

    meta_row = sample_meta_df.set_index("sample_id").loc[sample_id]

    for col in sample_meta_df.columns:
        if col == "sample_id":
            continue
        adata.obs[col] = meta_row[col]
    #adata.write(adata_init_path)

    return adata

adata_init = import_raw_data_csv(project_name, sample_id, original_data_dir,adata_init_path, sample_meta_df)
adata_init.obs


,sample_id,developmental_stage,sex,cell_type_descript
F_10W_embryo1_sc1,F_10W,10 week gestation,female,Primordial Germ Cells
F_10W_embryo1_sc2,F_10W,10 week gestation,female,Primordial Germ Cells
F_10W_embryo1_sc3,F_10W,10 week gestation,female,Primordial Germ Cells
F_10W_embryo1_sc4,F_10W,10 week gestation,female,Primordial Germ Cells
F_10W_embryo1_sc5,F_10W,10 week gestation,female,Primordial Germ Cells
...,...,...,...,...
F_10W_embryo1_sc90,F_10W,10 week gestation,female,Primordial Germ Cells
F_10W_embryo1_sc91,F_10W,10 week gestation,female,Primordial Germ Cells
F_10W_embryo1_sc92,F_10W,10 week gestation,female,Primordial Germ Cells
F_10W_embryo1_sc93,F_10W,10 week gestation,female,Primordial Germ Cells


In [27]:
adata_init = import_raw_data_csv(project_name, sample_id, original_data_dir,adata_init_path, sample_meta_df)

Empty DataFrame
Columns: []
Index: [F_10W_embryo1_sc1, F_10W_embryo1_sc2, F_10W_embryo1_sc3, F_10W_embryo1_sc4, F_10W_embryo1_sc5, F_10W_embryo1_sc6, F_10W_embryo1_sc7, F_10W_embryo1_sc8, F_10W_embryo1_sc9, F_10W_embryo1_sc10, F_10W_embryo1_sc11, F_10W_embryo1_sc12, F_10W_embryo1_sc13, F_10W_embryo1_sc14, F_10W_embryo1_sc15, F_10W_embryo1_sc16, F_10W_embryo1_sc17, F_10W_embryo1_sc18, F_10W_embryo1_sc19, F_10W_embryo1_sc20, F_10W_embryo1_sc21, F_10W_embryo1_sc22, F_10W_embryo1_sc23, F_10W_embryo1_sc24, F_10W_embryo1_sc25, F_10W_embryo1_sc26, F_10W_embryo1_sc27, F_10W_embryo1_sc28, F_10W_embryo1_sc29, F_10W_embryo1_sc30, F_10W_embryo1_sc31, F_10W_embryo1_sc32, F_10W_embryo1_sc33, F_10W_embryo1_sc34, F_10W_embryo1_sc35, F_10W_embryo1_sc36, F_10W_embryo1_sc37, F_10W_embryo1_sc38, F_10W_embryo1_sc39, F_10W_embryo1_sc40, F_10W_embryo1_sc41, F_10W_embryo1_sc42, F_10W_embryo1_sc43, F_10W_embryo1_sc44, F_10W_embryo1_sc45, F_10W_embryo1_sc46, F_10W_embryo1_sc47, F_10W_embryo1_sc48, F_10W_embryo1

KeyError: 'sample_id'

for file in

# Whole project

In [5]:

# Collapse to sample x gene
sample_gene_df = (
    df.groupby(["sample_id", "gene"])
    .apply(lambda x: pd.Series({
        "n_cells": x["n_cells"].sum(),
        "n_positive_cells": x["n_positive_cells"].sum(),
        "frac_positive_cells": x["n_positive_cells"].sum() / x["n_cells"].sum(),
        "mean_expression": (x["mean_expression"] * x["n_cells"]).sum() / x["n_cells"].sum()
    }))
    .reset_index()
)

# Pick genes
top_n = 25

gene_order = (
    sample_gene_df.groupby("gene")["mean_expression"]
    .max()
    .sort_values(ascending=False)
    .index
)

gene_order_top = gene_order[:top_n]

plot_df = sample_gene_df[sample_gene_df["gene"].isin(gene_order_top)].copy()

# Set ordering
plot_df["gene"] = pd.Categorical(
    plot_df["gene"],
    categories=gene_order_top,
    ordered=True
)

# Cluster samples - optional but useful
from scipy.cluster.hierarchy import linkage, leaves_list

cluster_df = (
    plot_df.pivot(index="sample_id", columns="gene", values="mean_expression")
    .fillna(0)
)

Z = linkage(cluster_df.values, method="average", metric="correlation")
sample_order = cluster_df.index[leaves_list(Z)]

plot_df["sample_id"] = pd.Categorical(
    plot_df["sample_id"],
    categories=sample_order,
    ordered=True
)

/tmp/ipykernel_3407598/1545683502.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


In [43]:
for project_name in ["embryos_mixed"]:
    df = pd.read_csv(f"all_results/tables/cta_genes/{project_name}_cta_gene_expression.csv")

    dotplot_save_dir = "all_results/figures/cta_genes"
    os.makedirs(dotplot_save_dir, exist_ok = True)
    df = pd.read_csv(f"all_results/tables/cta_genes/{project_name}_cta_gene_expression.csv")
    # Collapse to sample x gene
    sample_gene_df = (
        df.groupby(["sample_id", "gene"])
        .apply(lambda x: pd.Series({
            "n_cells": x["n_cells"].sum(),
            "n_positive_cells": x["n_positive_cells"].sum(),
            "frac_positive_cells": x["n_positive_cells"].sum() / x["n_cells"].sum(),
            "mean_expression": (x["mean_expression"] * x["n_cells"]).sum() / x["n_cells"].sum()
        }))
        .reset_index()
    )

    # Pick genes
    top_n = 25

    gene_order = (
        sample_gene_df.groupby("gene")["mean_expression"]
        .max()
        .sort_values(ascending=False)
        .index
    )

    gene_order_top = gene_order[:top_n]

    plot_df = sample_gene_df[sample_gene_df["gene"].isin(gene_order_top)].copy()

    # Set ordering
    plot_df["gene"] = pd.Categorical(
        plot_df["gene"],
        categories=gene_order_top,
        ordered=True
    )

    # Cluster samples - optional but useful
    from scipy.cluster.hierarchy import linkage, leaves_list

    cluster_df = (
        plot_df.pivot(index="sample_id", columns="gene", values="mean_expression")
        .fillna(0)
    )

    Z = linkage(cluster_df.values, method="average", metric="correlation")
    sample_order = cluster_df.index[leaves_list(Z)]

    plot_df["sample_id"] = pd.Categorical(
        plot_df["sample_id"],
        categories=sample_order,
        ordered=True
    )

    import matplotlib.pyplot as plt
    import seaborn as sns

    plt.figure(figsize=(len(gene_order_top) * 0.3, len(sample_order) * 0.5))

    sns.scatterplot(
        data=plot_df,
        x="gene",
        y="sample_id",
        size="frac_positive_cells",
        hue="mean_expression",
        sizes=(10, 200),
        palette="Purples",
        edgecolor="none",
        hue_norm=(0, plot_df["mean_expression"].quantile(0.95))
    )

    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(fontsize=9)

    plt.xlabel("")
    plt.ylabel("")
    plt.title(f"{project_name}\nCTA Gene Expression Across Samples")

    plt.legend([], [], frameon=False)

    plt.tight_layout()
    plt.savefig(os.path.join(dotplot_save_dir, f"{project_name}_cta_all_samples_dotplot.png"),dpi=300, bbox_inches="tight")
    plt.close()

/tmp/ipykernel_3407598/1315011400.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


# Dotplots etc

In [4]:
project_name = "ovarian_cancer_ccca"
figures_dir = f"datasets/{project_name}/results/figures"
tables_dir = f"datasets/{project_name}/results/tables"
save_name = "cta_genes"
cta_genes_file_path = "reference_data_common/CTA_family.csv"
#dotplot_save_dir = os.path.join(figures_dir, save_name, "celltype_expression")
#table_save_dir = os.path.join(tables_dir, save_name, "dotplot_celltype")

#os.makedirs(dotplot_save_dir, exist_ok=True)
#os.makedirs(table_save_dir, exist_ok=True)

df = pd.read_csv(cta_genes_file_path)
df.columns = ["gene_group", "gene"]


all_genes_list = sorted(df["gene"].unique())
all_genes_list

['ACRBP',
 'ACTL8',
 'ADAM2',
 'ADAM29',
 'AKAP3',
 'AKAP4',
 'ANKRD45',
 'ARMC3',
 'ARX',
 'ATAD2',
 'BAGE',
 'BAGE2',
 'BAGE3',
 'BAGE4',
 'BAGE5',
 'BRDT',
 'C15orf60',
 'C21orf99',
 'CABYR',
 'CAGE1',
 'CALR3',
 'CASC5',
 'CCDC110',
 'CCDC33',
 'CCDC36',
 'CCDC62',
 'CCDC83',
 'CDCA1',
 'CEP290',
 'CEP55',
 'COX6B2',
 'CPXCR1',
 'CRISP2',
 'CSAG1',
 'CSAG2',
 'CSAG3B',
 'CT16.2',
 'CT45A1',
 'CT45A2',
 'CT45A3',
 'CT45A4',
 'CT45A5',
 'CT45A6',
 'CT47A1',
 'CT47A10',
 'CT47A11',
 'CT47A2',
 'CT47A3',
 'CT47A4',
 'CT47A5',
 'CT47A6',
 'CT47A7',
 'CT47A8',
 'CT47A9',
 'CT47B1',
 'CT66',
 'CT69',
 'CT70',
 'CTAG1A',
 'CTAG1B',
 'CTAG2',
 'CTAGE-2',
 'CTAGE1',
 'CTAGE5',
 'CTCFL',
 'CTNNA2',
 'CXorf48',
 'Cxorf61',
 'DCAF12',
 'DDX43',
 'DDX53',
 'DKKL1',
 'DMRT1',
 'DNAJB8',
 'DPPA2',
 'DSCR8',
 'EDAG, NDR',
 'ELOVL4',
 'FAM133A',
 'FAM46D',
 'FATE1',
 'FBXO39',
 'FMR1NB',
 'FTHL17',
 'GAGE1',
 'GAGE12B',
 'GAGE12C',
 'GAGE12D',
 'GAGE12E',
 'GAGE12F',
 'GAGE12G',
 'GAGE12H',
 'GAGE12

In [5]:
working_adata_dir = "datasets/ovarian_cancer_ccca/working_adata"
adata_annotated_list = os.listdir(working_adata_dir)

for file in adata_annotated_list:
    adata = sc.read_h5ad(os.path.join(working_adata_dir,file))


KeyboardInterrupt: 

In [16]:
sub_list = ['Geistlinger2020','Izar2020', 'Nath2021','Olalekan2021','Olbrecht2021','Qian2020','Regner2021','Shih2018','Tang-Huau2018','Zhang2019', 'Zhang2022']

for subproject in sub_list:
    save_dir = os.path.join(tables_dir, save_name, subproject)
    os.makedirs(save_dir, exist_ok = True) 
    all_expression_save_path = os.path.join(save_dir, f"{subproject}_{save_name}_all_expression.csv")
    
    adata = sc.read_h5ad(f"datasets/ovarian_cancer_ccca/working_adata/{subproject}_annotated_cellmarker.h5ad")

    rows = []

    for gene in all_genes_list:

        if gene not in adata.var_names:
            continue

        expr = adata[:, gene].X
        if hasattr(expr, "toarray"):
            expr = expr.toarray().flatten()
        else:
            expr = expr.flatten()

        for cell_type in adata.obs["predicted_cell_type"].unique():
        
            mask = adata.obs["predicted_cell_type"] == cell_type
            expr_ct = expr[mask.values]

            n_ct = mask.sum()
            n_pos = (expr_ct > 0).sum()
            mean_expr = expr_ct.mean() if n_ct > 0 else 0.0
            frac_pos = n_pos / n_ct if n_ct > 0 else 0.0

            rows.append({
                "sample_id": subproject,
                "predicted_cell_type": cell_type,
                "gene": gene,
                "n_cells": int(n_ct),
                "n_positive_cells": int(n_pos),
                "frac_positive_cells": float(frac_pos),
                "mean_expression": float(mean_expr),
            })

    df_save = pd.DataFrame(rows)
    df_save.to_csv(all_expression_save_path, index=False)



: 

In [12]:
df_save

,sample_id,predicted_cell_type,gene,n_cells,n_positive_cells,frac_positive_cells,mean_expression
0,Zhang2022,Epithelial cell,BRDT,2806,3,0.001069,-0.025035
1,Zhang2022,Pancreatic progenitor cell,BRDT,3410,0,0.000000,-0.028829
2,Zhang2022,Macrophage,BRDT,3970,0,0.000000,-0.028829
3,Zhang2022,Fibroblast,BRDT,1810,2,0.001105,-0.017748
4,Zhang2022,Plasma cell,BRDT,581,0,0.000000,-0.028829
...,...,...,...,...,...,...,...
177,Zhang2022,Plasmacytoid dendritic cell,PAGE5,303,0,0.000000,-0.037842
178,Zhang2022,Mast cell,PAGE5,627,0,0.000000,-0.037842
179,Zhang2022,Innate lymphoid cell,PAGE5,199,0,0.000000,-0.037842
180,Zhang2022,Microglial cell,PAGE5,212,1,0.004717,0.003442


In [ ]:
# For each gene and cell type, take the maximum frac_positive_cells --> one value per (gene, cell type) pair (collapses duplicates if they exist)
# Reshape to matrix (gene (rows) x celltype (columns) and fill na weith 0)
gene_dataset_max = (
        df_save.groupby(["gene", "predicted_cell_type"])["frac_positive_cells"]
        .max()
        .reset_index()
        .pivot(index="gene", columns="predicted_cell_type", values="frac_positive_cells")
        .fillna(0)
    )

# Removes genes where all cell types = 0, i.e. where genes are not expressed anywhere

gene_dataset_max = gene_dataset_max.loc[
        ~(gene_dataset_max == 0).all(axis=1)
    ]
# Sort genes by strongest signal - For each gene, take its maximum value across cell types, and Sort genes by that value (descending)
# genes most strongly expressed in any cell type appear first
gene_dataset_max = gene_dataset_max.loc[
        gene_dataset_max.max(axis=1).sort_values(ascending=False).index
    ]
# “For each gene, what is the strongest cell-type-specific signal it shows?”
gene_dataset_max

predicted_cell_type,Antigen-presenting cell,B cell,CD4 T cell,CD8 T cell,Cancer stem-like cell,Endothelial cell,Epithelial cell,Fibroblast,Germinal center cell,Granulosa cell,...,Microglial cell,Mucosa-associated invariant T (MAIT) cell,Neutrophil,Pancreatic progenitor cell,Plasma cell,Plasmacytoid dendritic cell,Sensory neuron,Sezary cell,T-cell lineage,Theca cell
gene,,,,,,,,,,,,,,,,,,,,,
IL13RA2,0.000840,0.000415,0.000000,0.0,0.000630,0.019802,0.009622,0.002762,0.000000,0.206681,...,0.000000,0.000000,0.000451,0.000880,0.001721,0.000000,0.124057,0.000000,0.000359,0.002669
CTCFL,0.000840,0.000000,0.000000,0.0,0.000420,0.000000,0.112259,0.005525,0.000000,0.002088,...,0.004717,0.000000,0.000451,0.068622,0.000000,0.000000,0.000754,0.000000,0.000000,0.000890
PAGE2,0.000840,0.000830,0.000000,0.0,0.000420,0.009901,0.080898,0.004972,0.008876,0.007307,...,0.000000,0.000468,0.001803,0.001173,0.012048,0.000000,0.018100,0.000903,0.001795,0.000000
CTAG2,0.002519,0.002906,0.010899,0.0,0.007144,0.019802,0.000356,0.004420,0.008876,0.004175,...,0.000000,0.000468,0.001353,0.000000,0.013769,0.009901,0.031297,0.001805,0.003230,0.001779
PAGE5,0.000000,0.000000,0.002725,0.0,0.001261,0.000000,0.016037,0.000552,0.008876,0.004175,...,0.004717,0.000936,0.000000,0.000880,0.000000,0.000000,0.000377,0.000000,0.003948,0.000000
BRDT,0.000000,0.000000,0.002725,0.0,0.003782,0.000000,0.001069,0.001105,0.005917,0.000000,...,0.000000,0.002808,0.000451,0.000000,0.000000,0.000000,0.000377,0.000000,0.000718,0.000000
MORC1,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.001069,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.003300,0.000000,0.000000,0.000000,0.000000


: 

In [ ]:


    plot_df = gene_dataset_max.head(top_n).T
    plot_df.index.name = "cell_type"
    plot_df.reset_index(inplace=True)

    plot_df_long = plot_df.melt(
        id_vars="cell_type",
        var_name="gene",
        value_name="frac_positive"
    )

    gene_order = gene_dataset_max.head(top_n).index.tolist()
    plot_df_long["gene"] = pd.Categorical(
        plot_df_long["gene"],
        categories=gene_order,
        ordered=True
    )

    n_groups = plot_df_long["cell_type"].nunique()

    from scipy.cluster.hierarchy import linkage, leaves_list

    # --- Prepare matrix ---
    cluster_df = gene_dataset_max.head(top_n)

    cluster_df = cluster_df.loc[cluster_df.max(axis=1) > 0.05]

    if cluster_df.empty:
        print("No strong genes to cluster.")
        return

    cluster_df = cluster_df.T

    cluster_values = np.log1p(cluster_df.values)

    cluster_values = np.nan_to_num(cluster_values)
    cluster_values = cluster_values[:, np.std(cluster_values, axis=0) > 0]
    cluster_values = cluster_values[np.std(cluster_values, axis=1) > 0]

    from scipy.cluster.hierarchy import linkage, leaves_list

    Z = linkage(cluster_values, method="average", metric="correlation")
    cell_order = cluster_df.index[leaves_list(Z)]

    # Apply clustered order to dotplot
    plot_df_long["cell_type"] = pd.Categorical(
        plot_df_long["cell_type"],
        categories=cell_order,
        ordered=True
    )

    
    # Clustered Heatmap
    

    g = sns.clustermap(
        cluster_df,
        cmap="Purples",
        row_cluster=True,
        col_cluster=False,  # genes already ranked
        linewidths=0.1,
        figsize=(top_n * 0.3, 8),
        cbar_kws={"label": "Fraction positive cells"}
    )

    g.fig.suptitle(
        f"{sample_id}\nCTA Expression by Cell Type",
        y=1.02
    )

    g.savefig(
        os.path.join(
            dotplot_save_dir,
            f"{sample_id}_{save_name}_celltype_clustered.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(g.fig)

    
    # Dotplot (clustered order)
    

    n_groups = len(cell_order)

    plt.figure(figsize=(top_n * 0.35, n_groups * 0.6))

    sns.scatterplot(
        data=plot_df_long,
        x="gene",
        y="cell_type",
        size="frac_positive",
        hue="frac_positive",
        sizes=(20, 300),
        palette="Purples"
    )

    plt.xticks(rotation=45, ha="right")
    plt.title(f"{sample_id}\nCTA Expression by Cell Type")

    handles, labels = plt.gca().get_legend_handles_labels()
    if handles:
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.savefig(
        os.path.join(
            dotplot_save_dir,
            f"{sample_id}_{save_name}_celltype_dotplot.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    return adata

# Import data

In [37]:
df_izar = pd.read_csv("datasets/ovarian_cancer_ccca/original_data/Data_Izar2020_Ovarian/Samples.csv")

df_nath = pd.read_csv("datasets/ovarian_cancer_ccca/original_data/Data_Nath2021_Ovarian/Samples.csv")


In [43]:
os.listdir("datasets/ovarian_cancer_ccca/original_data")

[file.split("_")[1] for file in os.listdir("datasets/ovarian_cancer_ccca/original_data")]

['Geistlinger2020',
 'Izar2020',
 'Nath2021',
 'Olalekan2021',
 'Olbrecht2021',
 'Qian2020',
 'Regner2021',
 'Shih2018',
 'Tang-Huau2018',
 'Zhang2019',
 'Zhang2022']

In [ ]:
    if subproject in ["Izar2020"]:
        matrix_path = os.path.join(original_data_dir, f"Data_{subproject}_Ovarian", "Exp_data_TPM.mtx")
    else:
        matrix_path = os.path.join(original_data_dir, f"Data_{subproject}_Ovarian", "Exp_data_UMIcounts.mtx")
    barcodes_path = os.path.join(original_data_dir, f"Data_{subproject}_Ovarian", "Cells.tsv")
    genes_path = os.path.join(original_data_dir, f"Data_{subproject}_Ovarian", "Genes.txt")
    

    X = mmread(matrix_path).tocsr().T

    genes = pd.read_csv(genes_path, header=None, sep="\t")
    barcodes = pd.read_csv(barcodes_path, header=None)

    adata = sc.AnnData(X)

    adata.var_names = genes[1].astype(str).values
    adata.obs_names = barcodes[0].astype(str).values

    print("var_names",adata.var_names[:10])
    print("obs_names", adata.obs_names[:10])

    adata.var_names_make_unique()

In [38]:


df_nath = df_nath.dropna(axis=1, how="all")

df_nath = df_nath.astype(str)

In [39]:
df_nath.to_csv("datasets/ovarian_cancer_ccca/original_data/Data_Nath2021_Ovarian/Samples copy.csv", index=False)

In [40]:
df_nath = pd.read_csv("datasets/ovarian_cancer_ccca/original_data/Data_Nath2021_Ovarian/Samples copy.csv")
df_nath 

,sample,technology,n_cells,patient,cancer_type,sex,diagnosis_recurrence,disease_extent,sample_primary_met,site,...,chemotherapy_exposed,chemotherapy_response,targeted_rx_exposed,targeted_rx_response,ICB_exposed,ICB_response,ET_exposed,ET_response,post_sampling_rx_exposed,OS
0,P02,iCell8,97,P02,Ovarian Cancer,Female,NaN,metastatic,met,ascites,...,carbo/cisplatin +paclitaxel\nliposomal doxorub...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,carbo/cisplatin + gemcitabine\ntopotecan\npacl...,1049.0
1,P03,iCell8,51,P03,Ovarian Cancer,Female,NaN,metastatic,met,ascites,...,carbo/cisplatin +paclitaxel\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,paclitaxel\nfarletuzumab,929.0
2,P04,10x,5989,P04,Ovarian Cancer,Female,NaN,metastatic,met,ascites,...,carbo/cisplatin +paclitaxel\nliposomal doxorub...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,carbo/cisplatin\nanastrozole\npaclitaxel\nlipo...,2011.0
3,P05,10x,8161,P05,Ovarian Cancer,Female,diagnosis,metastatic,met,ascites,...,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,carbo/cisplatin +paclitaxel,448.0
4,P10,10x,159,P10,Ovarian Cancer,Female,diagnosis,metastatic,met,ascites/plerual effusion,...,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,NaN,NaN
5,P11,10x,1437,P11,Ovarian Cancer,Female,diagnosis,metastatic,met,ascites/plerual effusion,...,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,NaN,NaN
6,P12,10x,1612,P12,Ovarian Cancer,Female,diagnosis,metastatic,met,ascites/plerual effusion,...,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,NaN,NaN
7,P13,10x,2065,P13,Ovarian Cancer,Female,diagnosis,metastatic,met,ascites/plerual effusion,...,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,NaN,NaN
8,P14,10x,2799,P14,Ovarian Cancer,Female,diagnosis,metastatic,met,ascites/plerual effusion,...,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,NaN,NaN
9,P15,10x,1057,P15,Ovarian Cancer,Female,diagnosis,metastatic,met,ascites/plerual effusion,...,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,not exposed,NaN,NaN


In [69]:
original_data_dir = "datasets/ovarian_cancer_ccca/original_data"
subproject = "Geistlinger2020"
matrix_path = os.path.join(original_data_dir, f"Data_{subproject}_Ovarian", "Exp_data_UMIcounts.mtx")
barcodes_path = os.path.join(original_data_dir, f"Data_{subproject}_Ovarian", "Cells.csv")
genes_path = os.path.join(original_data_dir, f"Data_{subproject}_Ovarian", "Genes.txt")
sample_meta_path = os.path.join(original_data_dir, f"Data_{subproject}_Ovarian", "Samples.csv")

X = mmread(matrix_path).tocsr().T

genes = pd.read_csv(genes_path, header=None, sep=None, engine="python")
barcodes = pd.read_csv(barcodes_path, sep=None, engine="python")

adata = sc.AnnData(X)

adata.var_names = genes.iloc[:, -1].astype(str).values
adata.obs_names = barcodes["cell_name"].astype(str).values

adata.obs = barcodes.set_index("cell_name")

sample_meta_df = pd.read_csv(sample_meta_path)
sample_meta_df = sample_meta_df.dropna(axis=1, how="all")
sample_meta_df = sample_meta_df.astype(str)

# columns to add (exclude overlaps except 'sample')
cols_to_add = [c for c in sample_meta_df.columns if c not in adata.obs.columns or c == "sample"]

adata.obs = adata.obs.reset_index().merge(sample_meta_df[cols_to_add], on="sample", how="left").set_index("cell_name")
adata.obs = adata.obs.rename(columns={"sample": "sample_id"})

adata.obs

,sample_id,patient,cell_type,cell_subtype,complexity,umap1,umap2,g1s_score,g2m_score,cell_cycle_phase,...,hpca.celltype,encode.celltype,subtype,tumor_stage,tumor_grade,ct_response,cancer_type,technology,n_cells,histology
cell_name,,,,,,,,,,,,,,,,,,,,,
AAACCTGAGCTGCCCA-1_1,T59,T59,Macrophage,Macrophage,799,NaN,NaN,NaN,NaN,NaN,...,Macrophage,Macrophages,DIF,IV,3,resistant,Ovarian Cancer,10x,12659,HGSOC
AAACCTGAGTCATCCA-1_2,T59,T59,Macrophage,Macrophage,1022,-20.1389,3.4806,-0.0742,-0.1413,Not cycling,...,Macrophage,Macrophages,DIF,IV,3,resistant,Ovarian Cancer,10x,12659,HGSOC
AAACCTGCAAGCCCAC-1_3,T59,T59,Macrophage,Macrophage,1036,-18.0940,5.6027,0.0662,0.1092,Not cycling,...,Macrophage,Macrophages,IMR,IV,3,resistant,Ovarian Cancer,10x,12659,HGSOC
AAACCTGCAAGCGCTC-1_4,T59,T59,Malignant,Malignant,2571,15.1243,-24.7119,0.8839,0.2127,G1/S,...,Epithelial_cells,Epithelial cells,DIF,IV,3,resistant,Ovarian Cancer,10x,12659,HGSOC
AAACCTGCACGTAAGG-1_5,T59,T59,Malignant,Malignant,3402,11.8044,-24.9339,0.8818,1.0527,Intermediate,...,Epithelial_cells,Epithelial cells,DIF,IV,3,resistant,Ovarian Cancer,10x,12659,HGSOC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCACACAGCCCA-1_41027,T90,T90,T_cell,T_cell,908,NaN,NaN,NaN,NaN,NaN,...,T_cells,CD8+ T-cells,DIF,IVa,3,sensitive,Ovarian Cancer,10x,3630,HGSOC
TTTGTCACATTCTTAC-1_41028,T90,T90,Malignant,Malignant,3321,-6.1218,-34.1287,2.3058,0.8404,G1/S,...,iPS_cells,Epithelial cells,DIF,IVa,3,sensitive,Ovarian Cancer,10x,3630,HGSOC
TTTGTCAGTAGCTGCC-1_41029,T90,T90,Fibroblast,Fibroblast,2583,11.6674,5.2157,-0.2370,-0.0576,Not cycling,...,Fibroblasts,Fibroblasts,MES,IVa,3,sensitive,Ovarian Cancer,10x,3630,HGSOC


In [68]:
adata.obs

,sample,patient,cell_type,cell_subtype,complexity,umap1,umap2,g1s_score,g2m_score,cell_cycle_phase,mp_top_score,mp_top,mp_assignment,hpca.celltype,encode.celltype,subtype,tumor_stage,tumor_grade,ct_response
cell_name,,,,,,,,,,,,,,,,,,,
AAACCTGAGCTGCCCA-1_1,T59,T59,Macrophage,Macrophage,799,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Macrophage,Macrophages,DIF,IV,3,resistant
AAACCTGAGTCATCCA-1_2,T59,T59,Macrophage,Macrophage,1022,-20.1389,3.4806,-0.0742,-0.1413,Not cycling,0.6188,MAC3,NaN,Macrophage,Macrophages,DIF,IV,3,resistant
AAACCTGCAAGCCCAC-1_3,T59,T59,Macrophage,Macrophage,1036,-18.0940,5.6027,0.0662,0.1092,Not cycling,0.4121,Lipid-associated,NaN,Macrophage,Macrophages,IMR,IV,3,resistant
AAACCTGCAAGCGCTC-1_4,T59,T59,Malignant,Malignant,2571,15.1243,-24.7119,0.8839,0.2127,G1/S,1.0352,Protein maturation,NaN,Epithelial_cells,Epithelial cells,DIF,IV,3,resistant
AAACCTGCACGTAAGG-1_5,T59,T59,Malignant,Malignant,3402,11.8044,-24.9339,0.8818,1.0527,Intermediate,1.6364,Cell Cycle - G2/M,Cell Cycle - G2/M,Epithelial_cells,Epithelial cells,DIF,IV,3,resistant
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCACACAGCCCA-1_41027,T90,T90,T_cell,T_cell,908,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T_cells,CD8+ T-cells,DIF,IVa,3,sensitive
TTTGTCACATTCTTAC-1_41028,T90,T90,Malignant,Malignant,3321,-6.1218,-34.1287,2.3058,0.8404,G1/S,2.4331,Cell Cycle - G1/S,Cell Cycle - G1/S,iPS_cells,Epithelial cells,DIF,IVa,3,sensitive
TTTGTCAGTAGCTGCC-1_41029,T90,T90,Fibroblast,Fibroblast,2583,11.6674,5.2157,-0.2370,-0.0576,Not cycling,2.0755,Complement,Complement,Fibroblasts,Fibroblasts,MES,IVa,3,sensitive


In [64]:
adata.obs

,sample,patient,cell_type,cell_subtype,complexity,umap1,umap2,g1s_score,g2m_score,cell_cycle_phase,mp_top_score,mp_top,mp_assignment,hpca.celltype,encode.celltype,subtype,tumor_stage,tumor_grade,ct_response,sample_id
cell_name,,,,,,,,,,,,,,,,,,,,
AAACCTGAGCTGCCCA-1_1,T59,T59,Macrophage,Macrophage,799,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Macrophage,Macrophages,DIF,IV,3,resistant,T59
AAACCTGAGTCATCCA-1_2,T59,T59,Macrophage,Macrophage,1022,-20.1389,3.4806,-0.0742,-0.1413,Not cycling,0.6188,MAC3,NaN,Macrophage,Macrophages,DIF,IV,3,resistant,T59
AAACCTGCAAGCCCAC-1_3,T59,T59,Macrophage,Macrophage,1036,-18.0940,5.6027,0.0662,0.1092,Not cycling,0.4121,Lipid-associated,NaN,Macrophage,Macrophages,IMR,IV,3,resistant,T59
AAACCTGCAAGCGCTC-1_4,T59,T59,Malignant,Malignant,2571,15.1243,-24.7119,0.8839,0.2127,G1/S,1.0352,Protein maturation,NaN,Epithelial_cells,Epithelial cells,DIF,IV,3,resistant,T59
AAACCTGCACGTAAGG-1_5,T59,T59,Malignant,Malignant,3402,11.8044,-24.9339,0.8818,1.0527,Intermediate,1.6364,Cell Cycle - G2/M,Cell Cycle - G2/M,Epithelial_cells,Epithelial cells,DIF,IV,3,resistant,T59
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCACACAGCCCA-1_41027,T90,T90,T_cell,T_cell,908,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T_cells,CD8+ T-cells,DIF,IVa,3,sensitive,T90
TTTGTCACATTCTTAC-1_41028,T90,T90,Malignant,Malignant,3321,-6.1218,-34.1287,2.3058,0.8404,G1/S,2.4331,Cell Cycle - G1/S,Cell Cycle - G1/S,iPS_cells,Epithelial cells,DIF,IVa,3,sensitive,T90
TTTGTCAGTAGCTGCC-1_41029,T90,T90,Fibroblast,Fibroblast,2583,11.6674,5.2157,-0.2370,-0.0576,Not cycling,2.0755,Complement,Complement,Fibroblasts,Fibroblasts,MES,IVa,3,sensitive,T90
